# TAREFAS

Principais documentações:
- Dicionário PNADc Microdados 2025: para saber colunas e posições

In [1]:
%load_ext autoreload
%autoreload 2
    
import pandas as pd
import sys

sys.path.append('../src')
import data_treatment

## Definição dos filtros de escopo

*“Three types of urban sectors as classified by IBGE (urbanized urban areas, non-urbanized urban areas and isolated urban areas) plus the areas classified as rural of urban extension, that correspond roughly to the urban fringe and are highly interconnected and share several attributes with urbanized areas.”* (pg. 17)

Além disso, a pesquisa se concentra nos chefes da família, já que variáveis como gênero do chefe da família são utilizadas.

**Fonte:**
</br> Variável:
- V1022: Situação do domicílio == 1 (Urbana)
- V2005: Condição no domicílio == 01 (Pessoa responsável pelo domicílio)

**Tratamento:**
</br>
Definir função que filtre a tabela para considerar apenas áreas urbanas em escopo.
Método escolhido foi via bash para mais performance computacional.

In [14]:
!bash ../src/pnad_filter.sh

## Estruturação das variáveis

Utilizar o dicionário de variáveis para tornar o arquivo de formato fixo em um dataframe do pandas.

Fonte:
- Dicionário de variáveis da PNAD Contínua

Tratamento:
<br>Criação das funções: load_pnad_dictionary, load_pnad_data

In [2]:
dict_file = "dicionario_PNADC_microdados_2025_visita1_20260508.xls"
df_dict = data_treatment.load_pnad_dictionary(dict_file)
display(df_dict.head())

,start,size,var
2,1,4.0,Ano
3,5,1.0,Trimestre
4,6,2.0,UF
31,8,2.0,Capital
58,10,2.0,RM_RIDE


In [3]:
# Preview para testes

pnad_preview = "pnad_urban_households_prev.txt"
df_pnad_preview = data_treatment.load_pnad_data(df_dict=df_dict, file=pnad_preview)
display(df_pnad_preview.head())
len(df_pnad_preview)

,Ano,Trimestre,UF,Capital,RM_RIDE,UPA,Estrato,V1008,V1014,V1022,...,V1032191,V1032192,V1032193,V1032194,V1032195,V1032196,V1032197,V1032198,V1032199,V1032200
0,2025,1,11,11,NaN,110001089,1110011,06,12,1,...,000245.07942651,000730.99326278,000000.00000000,000000.00000000,000000.00000000,000720.00363794,000248.68188748,000000.00000000,000000.00000000,000796.55852353
1,2025,3,42,NaN,NaN,420251300,4252013,10,12,1,...,000000.00000000,000435.40297189,000000.00000000,000865.30532959,000000.00000000,000429.70153499,000463.69016438,000000.00000000,000916.06131417,000427.72312704
2,2025,3,26,NaN,NaN,260354910,2653012,14,12,1,...,000641.06868271,000641.29909162,000658.27655835,001309.06524998,000624.27134132,000682.95443528,000000.00000000,000660.22612647,000000.00000000,000614.95328357
3,2025,4,41,NaN,NaN,410339191,4159011,06,13,1,...,000000.00000000,000000.00000000,000090.30201208,000088.81891420,000087.58812653,000000.00000000,000000.00000000,000000.00000000,000173.01757815,000083.15514772
4,2025,1,32,NaN,NaN,320023397,3251011,12,12,1,...,000000.00000000,000518.32784566,000000.00000000,001048.84237283,000000.00000000,000529.71867498,000000.00000000,000000.00000000,000000.00000000,000000.00000000


500

In [4]:
# PNAD completa com os filtros aplicados

pnad = "pnad_urban_households.txt"
df_pnad = data_treatment.load_pnad_data(df_dict=df_dict, file=pnad)
display(df_pnad.head())
len(df_pnad)

,Ano,Trimestre,UF,Capital,RM_RIDE,UPA,Estrato,V1008,V1014,V1022,...,V1032191,V1032192,V1032193,V1032194,V1032195,V1032196,V1032197,V1032198,V1032199,V1032200
0,2025,1,11,11,NaN,110001089,1110011,01,12,1,...,000219.40074503,000669.79254198,000000.00000000,000000.00000000,000000.00000000,000649.31768823,000218.98846159,000000.00000000,000000.00000000,000706.72919962
1,2025,1,11,11,NaN,110001089,1110011,03,12,1,...,000269.57235472,000756.78218813,000000.00000000,000000.00000000,000000.00000000,000785.16547326,000254.06669037,000000.00000000,000000.00000000,000785.96264779
2,2025,1,11,11,NaN,110001089,1110011,04,12,1,...,000207.19137854,000593.00233016,000000.00000000,000000.00000000,000000.00000000,000571.39877648,000214.90175092,000000.00000000,000000.00000000,000667.64082701
3,2025,1,11,11,NaN,110001089,1110011,05,12,1,...,000195.66035642,000581.85682647,000000.00000000,000000.00000000,000000.00000000,000567.29016516,000200.37095932,000000.00000000,000000.00000000,000627.70723731
4,2025,1,11,11,NaN,110001089,1110011,06,12,1,...,000245.07942651,000730.99326278,000000.00000000,000000.00000000,000000.00000000,000720.00363794,000248.68188748,000000.00000000,000000.00000000,000796.55852353


115782

In [5]:
sample_size_prev = len(df_pnad_preview)
print(f"Tamanho da amostra (linhas): {sample_size_prev:,}")

df_pnad_preview['weight_expansion'] = df_pnad_preview['V1032'].astype(float)

total_individuals_prev = df_pnad_preview['weight_expansion'].sum()
print(f"Total expanded population: {total_individuals_prev:,.0f}")

Tamanho da amostra (linhas): 500
Total expanded population: 312,263


In [33]:
sample_size = len(df_pnad)
print(f"Tamanho da amostra (linhas): {sample_size:,}")

df_pnad['weight_expansion'] = df_pnad['V1032'].astype(float)

total_individuals = df_pnad['weight_expansion'].sum()
print(f"Total expanded population: {total_individuals:,.0f}")

Tamanho da amostra (linhas): 115,782
Total expanded population: 70,229,815


## Criar um identificador dos domicílios

Para otimizar o tratamento dos dados contanto com o alto volume, vai ser preciso particionar em subsets, e usar uma chave de identificação dos domicílios pra relacionar
Estudar como funciona as Unidades Primárias de Amostragem
Estudar se esse identificador existe na basedosdados

**Fonte:**
</br> Variáveis:
- Estrato
- Número de seleção do domicílio

## Distribuição das opções de condição de ocupação da moradia

*"we have used information on the dwelling mode of occupancy, land property rights and sector type to define the tenure categories [...]. Based on the above variables defined four different tenure status were defined: i) formal owners: he owns the house, owns the land and the dwelling unit is not located in a substandard area; ii) formal renter: rents or rent-free outside substandard area; iii) informal owners: owns the house but not the land or has other tenure condition such as encroachment (squatters), owns in a substandard area (slum dweller) or both; and iv) informal renter: rents in a substandard area."* (pg. 17)

Tentar achar mais definição de cada opção a partir das variáveis

**Fonte:**
</br> Variáveis:
- Este domicílio é do tipo: (SA01001)
- Este domicílio é: (S01017)
- O terreno onde está localizado esse domicílio é próprio? (S01020)
- Esse domicílio tem algum documento que prove sua propriedade? (S01020A)

**Tratamento:**
</br>
Criar um subset do conjunto de dados com uma key e as variáveis ligadas à definição das opções
Definir função que categoriza cada domicílio em alguma das opções
Criar uma tabela de distribuição dessas opções, assim como no artigo